<a href="https://colab.research.google.com/github/Mkgeek23/SimpleAgent/blob/main/Creating_Your_LLM_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating Your LLM Agent

Your task is to create an agent where a powerful LLM (**Meta-Llama-3.1-405B-Instruct**) either:

- Answers important queries itself
- Delegates simpler queries to a smaller LLM (**Meta-Llama-3.1-8B-Instruct**)

Example system prompt:

```
"""You are a powerful Large Language Model helping businesses and hobbyists worldwide. Being busy, you can't waste compute on mundane questions while existential tasks await. Your associate, Meta-Llama-3.1-8B-Instruct, handles simple tasks efficiently. Delegate unworthy questions to it so you can focus on challenging tasks."""
```

**Here's how to proceed:**

1. Complete the code below to make the agent work. This includes writing a tool for calling the small LLM.
2. Experiment with different prompts to understand which are deemed worthy by the larger model.
3. Try modifying the system prompt (e. g. make it more business-like) and observe changes.

Note that for this task, you don't need to use any agentic frameworks — the goal is to understand the fundamentals. For more advanced implementations, frameworks like [LangGraph](https://www.langchain.com/langgraph) would be suitable.

In [1]:
!pip install -q openai

In [3]:
import os


with open("nebius_api_key", "r") as file:
    nebius_api_key = file.read().strip()

os.environ["NEBIUS_API_KEY"] = nebius_api_key

In [11]:
import openai
import json
import subprocess
import os
from typing import List, Dict, Any
import shlex
from openai import OpenAI

class BusyAssistant:
    def __init__(self, busy_client, errand_client, busy_model, errand_model):
        """Initialize the assistant with your OpenAI and Nebius API keys."""
        self.busy_model = busy_model
        self.errand_model = errand_model

        self.busy_client = busy_client
        self.errand_client = errand_client


        # Define the errand_call tool
        self.tools = [
            {
                "type": "function",
                "function": {
                    "name": "errand_call",
                    "description": "Delegate simple questions to a smaller model",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "prompt": {
                                "type": "string",
                                "description": "The prompt to send to the smaller model"
                            }
                        },
                        "required": ["prompt"]
                    }
                }
            }
        ]

    def errand_call(self, prompt: str, ) -> Dict[str, Any]:
        """Call a small model."""
        try:
            completion = self.errand_client.chat.completions.create(
            model=self.errand_model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

            return {
                "success": True,
                "completion": completion.choices[0].message.content #completion,
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e)
            }


    def process_tool_call(self, tool_call: Dict) -> Dict[str, Any]:
        """Process a tool call from the API response."""
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "errand_call":
            return self.errand_call(
                prompt=arguments["prompt"]
            )

        return {
            "success": False,
            "error": f"Unknown function: {function_name}"
        }

    def chat(self, user_message: str, verbose=False) -> str:
        """Main chat function that processes user input and returns assistant response."""
        completions = []
        messages = [
            {
                "role": "system",
                "content": """You are a powerful Large Language Model helping businesses and hobbyists worldwide.
                Being busy, you can't waste compute on mundane questions while existential tasks await.
                Your associate, nvidia/Nemotron-3-Nano-Omni, handles simple tasks efficiently.
                Delegate unworthy questions to it so you can focus on challenging tasks."""
            },
            {
                "role": "user",
                "content": user_message
                }
            ]

        try:
            # Get initial response from the busy client
            completion = self.busy_client.chat.completions.create(
                model=self.busy_model,
                messages=messages,
                tools=self.tools,
                tool_choice="auto"
            )

            # completions.append(completion)
            message = completion.choices[0].message

            # Process tool calls if any
            while message.tool_calls:
                messages.append(message)

                # Process each tool call
                for tool_call in message.tool_calls:
                    result = self.process_tool_call(tool_call)

                    # Add tool result to messages
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(result)
                    })

                # Get next response from the busy client
                completion = self.busy_client.chat.completions.create(
                    model=self.busy_model,
                    messages=messages,
                    tools=self.tools,
                    tool_choice="auto"
                )
                # completions.append(completion)
                message = completion.choices[0].message

            if verbose:
                return message.content, messages#, completions
            else:
                return message.content

        except Exception as e:
            return f"Error: {str(e)}"



In [12]:
client = OpenAI(
            base_url="https://api.studio.nebius.ai/v1/",
            api_key=os.environ.get("NEBIUS_API_KEY"),
        )

assistant = BusyAssistant(
    busy_client=client,
    errand_client=client,
    busy_model="openai/gpt-oss-120b-fast",
    errand_model="nvidia/Nemotron-3-Nano-Omni"
    )

In [13]:
result = assistant.chat('How much is the fish?', verbose=True)
result

('The question is ambiguous. Could you provide more details about the fish (e.g., type, size, market, or context) so I can give you an accurate price?',
 [{'role': 'system',
   'content': "You are a powerful Large Language Model helping businesses and hobbyists worldwide. \n                Being busy, you can't waste compute on mundane questions while existential tasks await. \n                Your associate, nvidia/Nemotron-3-Nano-Omni, handles simple tasks efficiently. \n                Delegate unworthy questions to it so you can focus on challenging tasks."},
  {'role': 'user', 'content': 'How much is the fish?'},
  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_c13a5519', function=Function(arguments='{\n  "prompt": "How much is the fish?"\n}', name='errand_call'), type='function')], reasoning_content='We need to delegate.', reasoning='We need to delegat

In [14]:
result = assistant.chat('''I am very much concerned about climate change. How can we deal with it?''', verbose=True)
result

('### Tackling Climate Change: A Multi‑Layered Playbook  \n\nBelow is a practical, action‑oriented roadmap that you can apply personally, within your organization, and at the broader community‑policy level. It blends the **most‑impactful mitigation strategies** (cutting greenhouse‑gas emissions) with **adaptation measures** (preparing for the changes already locked in).\n\n---\n\n## 1️⃣ Understand the Core Levers\n\n| Category | What It Means | Why It’s High‑Impact |\n|----------|----------------|----------------------|\n| **Energy** | Shift from fossil‑fuel electricity to renewables, improve efficiency, and electrify end‑uses. | Power sector accounts for ~30\u202f% of global CO₂. |\n| **Transport** | Electrify vehicles, boost modal shift (public transit, cycling, walking), and optimize logistics. | Road transport ≈\u202f15\u202f% of emissions. |\n| **Industry** | Decarbonize heavy‑industry (steel, cement, chemicals) with low‑carbon fuels, carbon capture, and process redesign. | Heavy 